# El Niño Research — Notebook 1: Data Exploration
**Author:** Srishti Chauhan | **Date:** May 2026

This notebook loads all raw datasets, checks their quality, and produces your first exploratory charts.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot styling
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_palette('deep')

RAW = '../data/raw/'
FIG = '../figures/'

import os
os.makedirs(FIG, exist_ok=True)
print('Libraries loaded successfully!')

## 1. Load All Datasets

In [ ]:
# --- ENSO / ONI Index ---
oni = pd.read_csv(RAW + 'oni_enso.csv')
print(f'ONI data: {oni.shape}  |  Columns: {list(oni.columns)}')
oni.head()

In [ ]:
# --- NBER Recession ---
nber = pd.read_csv(RAW + 'nber_recession.csv', parse_dates=['date'], index_col='date')
print(f'NBER data: {nber.shape}  |  Range: {nber.index.min()} to {nber.index.max()}')
nber.head()

In [ ]:
# --- Financial Markets ---
fin = pd.read_csv(RAW + 'financial_markets.csv', parse_dates=['date'], index_col='date')
print(f'Financial data: {fin.shape}')
fin.head()

In [ ]:
# --- Commodities ---
comm = pd.read_csv(RAW + 'commodities.csv', parse_dates=['date'], index_col='date')
print(f'Commodity data: {comm.shape}')
comm.head()

In [ ]:
# --- Crypto ---
crypto = pd.read_csv(RAW + 'crypto_prices.csv', parse_dates=['date'], index_col='date')
print(f'Crypto data: {crypto.shape}')
crypto.head()

---
## 2. El Niño Phase Analysis
How many seasons fall into each ENSO category?

In [ ]:
phase_counts = oni['phase'].value_counts()
print('ENSO Phase Distribution:')
print(phase_counts)

colors_map = {
    'Super El Nino':    '#C0392B',
    'Strong El Nino':   '#E74C3C',
    'Moderate El Nino': '#F39C12',
    'Weak El Nino':     '#F9CA24',
    'Neutral':          '#BDC3C7',
    'La Nina':          '#2980B9',
}

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(phase_counts.index,
              phase_counts.values,
              color=[colors_map.get(p, 'gray') for p in phase_counts.index],
              edgecolor='white', linewidth=0.5)
ax.set_title('Number of Seasons by ENSO Phase (1950–2025)', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of 3-Month Seasons')
ax.set_xlabel('ENSO Phase')
for bar, val in zip(bars, phase_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(val), ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(FIG + 'fig_enso_phase_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved!')

---
## 3. ONI Index Timeline
Identify the 5 key El Niño events we will study

In [ ]:
# Map 3-month seasons to approximate month numbers for plotting
season_to_month = {
    'DJF':1,'JFM':2,'FMA':3,'MAM':4,'AMJ':5,'MJJ':6,
    'JJA':7,'JAS':8,'ASO':9,'SON':10,'OND':11,'NDJ':12
}
oni['month'] = oni['season'].map(season_to_month)
oni['date']  = pd.to_datetime(dict(year=oni['year'], month=oni['month'], day=1))
oni_ts = oni.set_index('date').sort_index()

# Key events to annotate
events = {
    '1972-73 El Nino': '1972-11-01',
    '1982-83 Super':   '1982-11-01',
    '1997-98 Super':   '1997-10-01',
    '2015-16 Super':   '2015-10-01',
    '2023-24 El Nino': '2023-10-01',
}

fig, ax = plt.subplots(figsize=(16, 5))
ax.fill_between(oni_ts.index, oni_ts['ONI'], 0,
                where=oni_ts['ONI'] > 0, color='#E74C3C', alpha=0.4, label='El Nino')
ax.fill_between(oni_ts.index, oni_ts['ONI'], 0,
                where=oni_ts['ONI'] < 0, color='#2980B9', alpha=0.4, label='La Nina')
ax.plot(oni_ts.index, oni_ts['ONI'], color='#2C3E50', linewidth=0.8)
ax.axhline(2.0,  color='#922B21', linestyle='--', linewidth=1, label='Super El Nino threshold (+2.0)')
ax.axhline(0.5,  color='#E74C3C', linestyle=':',  linewidth=0.8)
ax.axhline(-0.5, color='#2980B9', linestyle=':',  linewidth=0.8)
ax.axhline(0,    color='black',   linestyle='-',  linewidth=0.5)

for label, date_str in events.items():
    dt = pd.to_datetime(date_str)
    if dt in oni_ts.index:
        val = oni_ts.loc[dt, 'ONI']
    else:
        val = 2.5
    ax.annotate(label, xy=(dt, val), xytext=(dt, val + 0.6),
                fontsize=7.5, ha='center', color='#2C3E50',
                arrowprops=dict(arrowstyle='->', color='gray', lw=0.8))

ax.set_title('Oceanic Nino Index (ONI) 1950–2025 | Key Super El Nino Events Highlighted',
             fontsize=13, fontweight='bold')
ax.set_ylabel('ONI Anomaly (deg C)')
ax.set_xlabel('Year')
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig(FIG + 'fig_oni_timeline.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 1 saved — this goes directly into your paper!')

---
## 4. S&P 500 vs El Niño Events

In [ ]:
if 'SP500' in fin.columns:
    sp = fin['SP500'].dropna()
    fig, ax = plt.subplots(figsize=(16, 5))
    ax.plot(sp.index, sp.values, color='#1A3A5C', linewidth=0.8, label='S&P 500')

    # Shade major El Nino events
    el_nino_periods = [
        ('1972-06-01', '1973-03-01', '1972-73'),
        ('1982-06-01', '1983-06-01', '1982-83 SUPER'),
        ('1997-04-01', '1998-04-01', '1997-98 SUPER'),
        ('2015-04-01', '2016-04-01', '2015-16 SUPER'),
        ('2023-06-01', '2024-03-01', '2023-24'),
    ]
    for start, end, lbl in el_nino_periods:
        ax.axvspan(pd.to_datetime(start), pd.to_datetime(end),
                   alpha=0.15, color='#E74C3C')
        mid = pd.to_datetime(start) + (pd.to_datetime(end) - pd.to_datetime(start))/2
        ax.text(mid, sp.max()*0.95, lbl, ha='center', fontsize=7,
                color='#922B21', fontweight='bold')

    ax.set_title('S&P 500 Price History with El Nino Event Windows Shaded',
                 fontsize=13, fontweight='bold')
    ax.set_ylabel('S&P 500 Index')
    ax.set_xlabel('Year')
    red_patch = mpatches.Patch(color='#E74C3C', alpha=0.3, label='El Nino Window')
    ax.legend(handles=[red_patch], fontsize=9)
    plt.tight_layout()
    plt.savefig(FIG + 'fig_sp500_el_nino.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Figure 2 saved!')
else:
    print('SP500 column not found — check financial_markets.csv')

---
## 5. Commodity Prices During El Niño Events

In [ ]:
if not comm.empty:
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    commodities_to_plot = [
        ('Wheat_Futures',  'Wheat',       '#8B4513', axes[0,0]),
        ('Corn_Futures',   'Corn',        '#DAA520', axes[0,1]),
        ('Coffee_Futures', 'Coffee',      '#6F4E37', axes[1,0]),
        ('CrudeOil_Futures','Crude Oil',  '#2C3E50', axes[1,1]),
    ]
    for col, name, color, ax in commodities_to_plot:
        if col in comm.columns:
            ax.plot(comm.index, comm[col], color=color, linewidth=0.8)
            for start, end, lbl in el_nino_periods:
                ax.axvspan(pd.to_datetime(start), pd.to_datetime(end),
                           alpha=0.15, color='#E74C3C')
            ax.set_title(f'{name} Futures Price', fontweight='bold', fontsize=10)
            ax.set_ylabel('Price (USD)')
        else:
            ax.set_title(f'{name} — no data', fontsize=10)

    fig.suptitle('Commodity Prices with El Nino Windows (Red Shading)',
                 fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(FIG + 'fig_commodities_el_nino.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Figure 3 saved!')

---
## 6. Bitcoin vs El Niño (2013–2025)

In [ ]:
if 'BTC_price' in crypto.columns:
    btc = crypto['BTC_price'].dropna()
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(btc.index, btc.values, color='#F7931A', linewidth=0.8, label='Bitcoin (USD)')
    for start, end, lbl in el_nino_periods:
        s, e = pd.to_datetime(start).date(), pd.to_datetime(end).date()
        if e > btc.index.min():
            ax.axvspan(s, e, alpha=0.15, color='#E74C3C')
            mid = s + (e - s)/2
            ax.text(mid, btc.max()*0.9, lbl, ha='center', fontsize=7,
                    color='#922B21', fontweight='bold')
    ax.set_title('Bitcoin Price (USD) with El Nino Event Windows',
                 fontsize=13, fontweight='bold')
    ax.set_ylabel('BTC Price (USD)')
    ax.set_xlabel('Year')
    plt.tight_layout()
    plt.savefig(FIG + 'fig_btc_el_nino.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Figure 4 saved!')
else:
    print('BTC data not found — check crypto_prices.csv')

---
## 7. Data Quality Summary

In [ ]:
print('='*55)
print('  DATA QUALITY SUMMARY')
print('='*55)
datasets = {
    'ONI / ENSO':         oni,
    'NBER Recession':     nber,
    'Financial Markets':  fin,
    'Commodities':        comm,
    'Cryptocurrency':     crypto,
}
for name, df in datasets.items():
    nulls = df.isnull().sum().sum()
    print(f'  {name:<22s}  Rows: {len(df):>6,}  |  Nulls: {nulls:>5,}  |  Cols: {len(df.columns)}')
print('='*55)
print('\nAll done! Move on to Notebook 02 for historical analysis.')